In [1]:
import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
# Helpful statement for debugging, prints the thing entered as x and the output, i.e.,
# debugPrint(1+1) will output '1+1 [int] = 2'
%matplotlib widget
import inspect
import re
def debugPrint(x):
    frame = inspect.currentframe().f_back
    s = inspect.getframeinfo(frame).code_context[0]
    r = re.search(r"\((.*)\)", s).group(1)
    print("{} [{}] = {}".format(r,type(x).__name__, x))
    
    
import os, sys
import torch
import numpy as np

# Warp-based Radius Search Implementation
import warp as wp

# Initialize Warp
wp.init()

from sphWarpCore.math import *
from sphWarpCore.util import castTorchToWarp, castWarpToTorch, castTorchToWarpAsBuiltins
from sphWarpCore.radiusSearch.wp_radius_small import warp_radius_search_small
from sphWarpCore.radius import *
from warp.types import vector
from sphWarpCore.autograd import warpWrapper, WarpFunctionWrapper
from sphWarpCore.ops import *
import matplotlib.pyplot as plt

wp.config.verify_autograd_array_access = True
wp.config.verbose = True


Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX PRO 500 Blackwell Generation Laptop GPU" (6 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/lu26029/.cache/warp/1.12.0


In [2]:
@torch.jit.script
def volumeToSupport(volume : float, targetNeighbors : int, dim : int):
    """
    Calculates the support radius based on the given volume, target number of neighbors, and dimension.

    Parameters:
    volume (float): The volume of the support region.
    targetNeighbors (int): The desired number of neighbors.
    dim (int): The dimension of the space.

    Returns:
    torch.Tensor: The support radius.
    """
    if dim == 1:
        # N_h = 2 h / v -> h = N_h * v / 2
        return targetNeighbors * volume / 2
    elif dim == 2:
        # N_h = \pi h^2 / v -> h = \sqrt{N_h * v / \pi}
        return torch.sqrt(targetNeighbors * volume / np.pi)
    else:
        # N_h = 4/3 \pi h^3 / v -> h = \sqrt[3]{N_h * v / \pi * 3/4}
        return torch.pow(targetNeighbors * volume / np.pi * 3 /4, 1/3)
    
def generateNeighborTestData(nx, targetNumNeighbors, dim, periodic, device):


    minDomain = torch.tensor([-1] * dim, dtype = torch.float32, device = device)
    maxDomain = torch.tensor([ 1] * dim, dtype = torch.float32, device = device)
    periodicity = torch.tensor([periodic] * dim, device = device, dtype = torch.bool)

    extent = maxDomain - minDomain
    shortExtent = torch.min(extent, dim = 0)[0].item()
    dx = (shortExtent / nx)
    ny = int(1 // dx)
    h = volumeToSupport(dx**dim, targetNumNeighbors, dim)
    dy = dx / 1.5
    ny = int(1 // dy)

    positions = []
    for d in range(dim):
        positions.append(torch.linspace(minDomain[d] + dx / 2, maxDomain[d] - dx / 2, int((extent[d] - dx) / dx) + 1, device = device))
    grid = torch.meshgrid(*positions, indexing = 'xy')
    
    positions = torch.stack(grid, dim = -1).reshape(-1,dim).to(device)
    supports = torch.ones(positions.shape[0], device = device) * h
    
    domain = DomainDescription(minDomain, maxDomain, periodicity, dim)
    
    return positions, supports, positions.shape[0], domain, dx

In [3]:
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# if platform.system() == 'Darwin':
    # device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
device = torch.device('cpu')
device = torch.device('cuda')
targetNumNeighbors = 50
nx = 32
dim = 2
numParticles = nx**dim

def getNextPrime(n):
    # Compute the next larger prime number greater than n
    # used primarily to set the hash map length to a prime number for better distribution of particles in the hash map
    
    def is_prime(num):
        if num <= 1:
            return False
        for i in range(2, int(num**0.5) + 1):
            if num % i == 0:
                return False
        return True
    
    prime = n + 1
    while True:
        if is_prime(prime):
            return prime
        prime += 1

# hashMapLength = getNextPrime(numParticles)
# periodic = False
# x, h, numParticles, domain, dx = generateNeighborTestData(nx, targetNumNeighbors, dim, False, device)

# pointCloud = PointCloud(x, h)
# queryPositions = pointCloud.positions
# querySupports = pointCloud.supports
# referencePositions = pointCloud.positions
# referenceSupports = pointCloud.supports
# mode = 'gather'

# offset = 0

In [4]:

device = torch.device(device)
numParticles = nx**dim
hashMapLength = getNextPrime(numParticles)
periodic = False
x, h, numParticles, domain, dx = generateNeighborTestData(nx, targetNumNeighbors, dim, False, device)

x += torch.randn_like(x) * dx * 0.1

pointCloud = PointCloud(x, h)
queryPositions = pointCloud.positions.contiguous()
querySupports = pointCloud.supports
referencePositions = pointCloud.positions.contiguous()
referenceSupports = pointCloud.supports

queryMasses = torch.ones(x.shape[0], device = x.device) * dx**dim
referenceMasses = torch.ones(x.shape[0], device = x.device) * dx**dim

mode = 'gather'

In [5]:
import diffSPH
from diffSPH.sampling import ParticleSet
from diffSPH.schemes.states.common import BasicState
from diffSPH.modules.density import computeDensity
from diffSPH.neighborhood import PointCloud, DomainDescription, buildNeighborhood, filterNeighborhood, coo_to_csrsc, coo_to_csr
from diffSPH.kernels import *
from diffSPH.neighborhood import evaluateNeighborhood, SupportScheme, computeNeighborhoodStates
from diffSPH.enums import Operation, SupportScheme, GradientMode, LaplacianMode
from diffSPH.operations import SPHOperation
from sphWarpCore.ops import sphOperation_warp
from sphWarpCore.enumTypes import *
from sphWarpCore.operations.wp_covariance import computeSPHCovariance_warpBackend
from diffSPH.math import pinv2x2
from diffSPH.modules.renorm import computeCovarianceMatrices
from diffSPH.operations import KernelCorrectionScheme
from diffSPH.modules.adaptiveSmoothing import *
import diffSPH
from diffSPH.sampling import ParticleSet
from diffSPH.schemes.states.common import BasicState
from diffSPH.modules.density import computeDensity
from diffSPH.neighborhood import PointCloud, DomainDescription, buildNeighborhood, filterNeighborhood, coo_to_csrsc, coo_to_csr
from diffSPH.kernels import *
from diffSPH.neighborhood import evaluateNeighborhood, SupportScheme, computeNeighborhoodStates
from diffSPH.enums import Operation, SupportScheme, GradientMode, LaplacianMode
from diffSPH.operations import SPHOperation





In [6]:
import time
def timeFunction(func, *args, **kwargs):
    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize()
    cpuBegin = time.time()
    begin.record()
    
    result = func(*args, **kwargs)
    
    end.record()
    torch.cuda.synchronize()
    cpuEnd = time.time()
    gpuTime = begin.elapsed_time(end)
    cpuTime = cpuEnd - cpuBegin
    
    return result, gpuTime, cpuTime * 1000


In [7]:

def prepData(
    nx, targetNumNeighbors, dim, device, warpOnly = False
):

    device = torch.device(device)
    numParticles = nx**dim
    hashMapLength = getNextPrime(numParticles)
    periodic = False
    x, h, numParticles, domain, dx = generateNeighborTestData(nx, targetNumNeighbors, dim, False, device)

    x += torch.randn_like(x) * dx * 0.1

    pointCloud = PointCloud(x, h)
    queryPositions = pointCloud.positions.contiguous()
    querySupports = pointCloud.supports
    referencePositions = pointCloud.positions.contiguous()
    referenceSupports = pointCloud.supports

    queryMasses = torch.ones(x.shape[0], device = x.device) * dx**dim
    referenceMasses = torch.ones(x.shape[0], device = x.device) * dx**dim

    mode = 'gather'

    adjacency, adjacency_warp_gpu, adjacency_warp_cpu = timeFunction(radiusSearchCompactHashMap,
            queryPositions, referencePositions, 
            querySupports, referenceSupports, 
            domain.periodic, domain,
            mode, hashMapLength
    )
    
    densities = sphOperation_warp(
            queryPositions, referencePositions,
            querySupports, referenceSupports,
            queryMasses, referenceMasses,
            None, None,
            None, None,
            domain, adjacency,
            operation=WarpOperation.Density,
            kernel = KernelFunctions.Wendland2, supportMode = SupportScheme.Gather
    )
    measurement = {
        'numParticles': nx**dim,
        'targetNumNeighbors': targetNumNeighbors,
        'dim': dim,
        'device': device,
        'operation': 'Adjacency',
        'backend': 'warp',
        'gpuTime': adjacency_warp_gpu,
        'cpuTime': adjacency_warp_cpu        
    }

    if warpOnly == True:
        # print("Adjacency: ", adjacency)
        return queryPositions, referencePositions, querySupports, referenceSupports, queryMasses, referenceMasses, densities, densities, domain, adjacency, None, None, [measurement]

    particles_l = ParticleSet(positions = referencePositions, supports = referenceSupports, masses = referenceMasses, densities = torch.zeros_like(referenceMasses))

    positions_t = referencePositions.clone()
    simulationState = BasicState(
        positions = positions_t,
        supports = particles_l.supports,
        masses = particles_l.masses,
        densities = particles_l.densities,        
        velocities = torch.zeros_like(referencePositions),
        kinds = torch.zeros(referencePositions.shape[0], dtype = torch.int64, device = referencePositions.device),
        materials = torch.zeros(referencePositions.shape[0], dtype = torch.int64, device = referencePositions.device),
        UIDs = torch.arange(referencePositions.shape[0], device = referencePositions.device)
    )

    # neighborhood, neighbors = evaluateNeighborhood(simulationState, domain,  KernelType.Wendland4, verletScale =1.0, mode = SupportScheme.Gather, priorNeighborhood=None, computeHessian=False, computeDkDh=False, only_j = False)
    (neighborhood, sparseNeighborhood_), neighborhoodDiffSPHTime_gpu, neighborhoodDiffSPHTime_cpu  = timeFunction(buildNeighborhood, simulationState, simulationState, domain, verletScale = 1.0, mode = 'gather', priorNeighborhood=None)

    state, stateTime_gpu, stateTime_cpu = timeFunction(computeNeighborhoodStates, simulationState, sparseNeighborhood_, 'gather', KernelType.Wendland2, KernelType.Wendland2, True, True, False)

    neighborhood = state.get('noghost')

    measurement_diffSPH = {
        'numParticles': nx**dim,
        'targetNumNeighbors': targetNumNeighbors,
        'dim': dim,
        'device': device,
        'operation': 'Adjacency',
        'backend': 'diffSPH',
        'gpuTime': neighborhoodDiffSPHTime_gpu,
        'cpuTime': neighborhoodDiffSPHTime_cpu        
    }

    measurement_diffSPH_state = {
        'numParticles': nx**dim,
        'targetNumNeighbors': targetNumNeighbors,
        'dim': dim,
        'device': device,
        'operation': 'State',
        'backend': 'diffSPH',
        'gpuTime': stateTime_gpu,
        'cpuTime': stateTime_cpu        
    }


    simulationState.densities = densities
    return queryPositions, referencePositions, querySupports, referenceSupports, queryMasses, referenceMasses, densities, densities, domain, adjacency, neighborhood, simulationState, [measurement, measurement_diffSPH, measurement_diffSPH_state]


In [8]:
nSamples = 8

In [9]:
from tqdm.autonotebook import tqdm

def benchmarkDensity(
        queryPositions, referencePositions, 
        querySupports, referenceSupports, 
        queryMasses, referenceMasses, 
        densities_warp, densities_diffSPH, 
        queryQuantity, referenceQuantity,
        domain, adjacency, neighborhood, simulationState,
        nx, targetNumNeighbors, dim, device, nSamples, warpOnly = False
):
    measurements = []
    for i in tqdm(range(nSamples), desc = f"Benchmarking [warp] with {nx**dim} particles, target neighbors: {targetNumNeighbors}, dim: {dim}, device: {device}", leave=False):
        _, gpuTime, cpuTime = timeFunction(
            sphOperation_warp,
            queryPositions, referencePositions,
            querySupports, referenceSupports,
            queryMasses, referenceMasses,
            densities_warp, densities_warp,
            queryQuantity, referenceQuantity,
            domain, adjacency,
            operation=WarpOperation.Density,
            kernel = KernelFunctions.Wendland2, supportMode = SupportScheme.Gather
        )
        measurements.append({
            'numParticles': nx**dim,
            'targetNumNeighbors': targetNumNeighbors,
            'dim': dim,
            'device': device,
            'operation': 'Density',
            'backend': 'warp',
            'gpuTime': gpuTime,
            'cpuTime': cpuTime
        })
    if warpOnly == True:
        return measurements
    for i in tqdm(range(nSamples), desc = f"Benchmarking [diffSPH] with {nx**dim} particles, target neighbors: {targetNumNeighbors}, dim: {dim}, device: {device}", leave=False):
        diffsph_result, diffsph_gpu_time, diffsph_cpu_time = timeFunction(SPHOperation,
                simulationState,
                quantity = referenceQuantity,
                kernel = KernelType.Wendland2,
                neighborhood = neighborhood[0],
                kernelValues = neighborhood[1],
                operation=Operation.Density,
                supportScheme = SupportScheme.Gather,
                correctionTerms= [],
                positiveDivergence=False
        )
        measurements.append({
            'numParticles': nx**dim,
            'targetNumNeighbors': targetNumNeighbors,
            'dim': dim,
            'device': device,
            'operation': 'Density',
            'backend': 'diffSPH',
            'gpuTime': diffsph_gpu_time,
            'cpuTime': diffsph_cpu_time
        })

    # for i in tqdm(range(nSamples), desc = f"Benchmarking [diffSPH] with {nx**dim} particles, target neighbors: {targetNumNeighbors}, dim: {dim}, device: {device}", leave=False):
    #     diffsph_result, diffsph_gpu_time, diffsph_cpu_time = timeFunction(SPHOperation,
    #             simulationState,
    #             quantity = referenceQuantity,
    #             kernel = KernelType.Wendland2,
    #             neighborhood = neighborhood[0],
    #             kernelValues = None,
    #             operation=Operation.Density,
    #             supportScheme = SupportScheme.Gather,
    #             correctionTerms= [],
    #             positiveDivergence=False
    #     )
    #     measurements.append({
    #         'numParticles': nx**dim,
    #         'targetNumNeighbors': targetNumNeighbors,
    #         'dim': dim,
    #         'device': device,
    #         'operation': 'Density',
    #         'backend': 'diffSPH [no pre]',
    #         'gpuTime': diffsph_gpu_time,
    #         'cpuTime': diffsph_cpu_time
    #     })
    return measurements


def warptodiffOperation(operation, gradientMode, laplacianMode):
    diffSPHOperation = None
    if operation == WarpOperation.Density:
        diffSPHOperation = Operation.Density
    elif operation == WarpOperation.Interpolate:
        diffSPHOperation = Operation.Interpolate
    elif operation == WarpOperation.Gradient:
        diffSPHOperation = Operation.Gradient
    elif operation == WarpOperation.Laplacian:
        diffSPHOperation = Operation.Laplacian
    elif operation == WarpOperation.Divergence:
        diffSPHOperation = Operation.Divergence
    elif operation == WarpOperation.Curl:
        diffSPHOperation = Operation.Curl
    else:
        raise ValueError(f"Unsupported operation type: {operation}")
    
    diffSPHGradientMode = None
    if gradientMode == GradientScheme.Naive:
        diffSPHGradientMode = GradientMode.Naive
    elif gradientMode == GradientScheme.Difference:
        diffSPHGradientMode = GradientMode.Difference
    elif gradientMode == GradientScheme.Summation:
        diffSPHGradientMode = GradientMode.Summation
    else:
        raise ValueError(f"Unsupported gradient mode: {gradientMode}")
    
    diffSPHLaplacianMode = None
    if laplacianMode == LaplacianScheme.Naive:
        diffSPHLaplacianMode = LaplacianMode.naive
    elif laplacianMode == LaplacianScheme.Brookshaw:
        diffSPHLaplacianMode = LaplacianMode.Brookshaw
    elif laplacianMode == LaplacianScheme.Dot:
        diffSPHLaplacianMode = LaplacianMode.dot
    elif laplacianMode == LaplacianScheme.Default:
        diffSPHLaplacianMode = LaplacianMode.default
    else:
        raise ValueError(f"Unsupported laplacian mode: {laplacianMode}")
    
    return diffSPHOperation, diffSPHGradientMode, diffSPHLaplacianMode

def benchmarkOperation(
        operation, gradientMode, laplacianMode,
        queryPositions, referencePositions, 
        querySupports, referenceSupports, 
        queryMasses, referenceMasses, 
        densities_warp, densities_diffSPH, 
        queryQuantity, referenceQuantity,
        domain, adjacency, neighborhood, simulationState,
        nx, targetNumNeighbors, dim, device, nSamples, warpOnly = False
):
    measurements = []
    for i in tqdm(range(nSamples), desc = f"Benchmarking [warp] with {nx**dim} particles, target neighbors: {targetNumNeighbors}, dim: {dim}, device: {device}", leave=False):
        _, gpuTime, cpuTime = timeFunction(
            sphOperation_warp,
            queryPositions, referencePositions,
            querySupports, referenceSupports,
            queryMasses, referenceMasses,
            densities_warp, densities_warp,
            queryQuantity, referenceQuantity,
            domain, adjacency,
            operation=operation, gradientMode=gradientMode, laplacianMode=laplacianMode,
            kernel = KernelFunctions.Wendland2, supportMode = SupportScheme.Gather
        )
        measurements.append({
            'numParticles': nx**dim,
            'targetNumNeighbors': targetNumNeighbors,
            'dim': dim,
            'device': device,
            'operation': operation.name,
            'backend': 'warp',
            'gpuTime': gpuTime,
            'cpuTime': cpuTime
        })
    if warpOnly == True:
        return measurements
    diffSPHOperation, diffSPHGradientMode, diffSPHLaplacianMode = warptodiffOperation(operation, gradientMode, laplacianMode)
    for i in tqdm(range(nSamples), desc = f"Benchmarking [diffSPH] with {nx**dim} particles, target neighbors: {targetNumNeighbors}, dim: {dim}, device: {device}", leave=False):
        diffsph_result, diffsph_gpu_time, diffsph_cpu_time = timeFunction(SPHOperation,
                simulationState,
                quantity = referenceQuantity,
                kernel = KernelType.Wendland2,
                neighborhood = neighborhood[0],
                kernelValues = neighborhood[1],
                operation=diffSPHOperation,
                gradientMode=diffSPHGradientMode,
                laplacianMode=diffSPHLaplacianMode,
                supportScheme = SupportScheme.Gather,
                correctionTerms= [],
                positiveDivergence=False
        )
        measurements.append({
            'numParticles': nx**dim,
            'targetNumNeighbors': targetNumNeighbors,
            'dim': dim,
            'device': device,
            'operation': diffSPHOperation.name,
            'backend': 'diffSPH',
            'gpuTime': diffsph_gpu_time,
            'cpuTime': diffsph_cpu_time
        })
        
    # diffSPHOperation, diffSPHGradientMode, diffSPHLaplacianMode = warptodiffOperation(operation, gradientMode, laplacianMode)
    # for i in tqdm(range(nSamples), desc = f"Benchmarking [diffSPH] with {nx**dim} particles, target neighbors: {targetNumNeighbors}, dim: {dim}, device: {device}", leave=False):
    #     diffsph_result, diffsph_gpu_time, diffsph_cpu_time = timeFunction(SPHOperation,
    #             simulationState,
    #             quantity = referenceQuantity,
    #             kernel = KernelType.Wendland2,
    #             neighborhood = neighborhood[0],
    #             kernelValues = None,
    #             operation=diffSPHOperation,
    #             gradientMode=diffSPHGradientMode,
    #             laplacianMode=diffSPHLaplacianMode,
    #             supportScheme = SupportScheme.Gather,
    #             correctionTerms= [],
    #             positiveDivergence=False
    #     )
    #     measurements.append({
    #         'numParticles': nx**dim,
    #         'targetNumNeighbors': targetNumNeighbors,
    #         'dim': dim,
    #         'device': device,
    #         'operation': diffSPHOperation.name,
    #         'backend': 'diffSPH [no pre]',
    #         'gpuTime': diffsph_gpu_time,
    #         'cpuTime': diffsph_cpu_time
    #     })
    return measurements

In [10]:

def benchmark(nx, targetNumNeighbors, dim, device, nSamples, warpOnly = False):
        with record_function(f"Total benchmark for {nx**dim} particles, target neighbors: {targetNumNeighbors}, dim: {dim}, device: {device}"):
            with record_function("Data Preparation"):
                queryPositions, referencePositions, \
                        querySupports, referenceSupports, \
                        queryMasses, referenceMasses, \
                        queryDensities, referenceDensities, \
                        domain, adjacency, neighborhood, simulationState, measurements = prepData(nx, targetNumNeighbors, dim, device, warpOnly)
            with record_function("Density Computation"):
                densityResults = benchmarkDensity(
                    queryPositions, referencePositions,
                    querySupports, referenceSupports,
                    queryMasses, referenceMasses,
                    queryDensities, referenceDensities,
                    None, None,
                    domain, adjacency, neighborhood, simulationState,
                    nx, targetNumNeighbors, dim, device, nSamples, warpOnly
                )
                measurements.extend(densityResults)


            f = torch.randn((queryPositions.shape[0],), device = queryPositions.device, dtype = queryPositions.dtype)
            with record_function("Interpolation Operations"):
                interpolationResults = benchmarkOperation(
                    WarpOperation.Interpolate, GradientScheme.Naive, LaplacianScheme.Naive,
                    queryPositions, referencePositions,
                    querySupports, referenceSupports,
                    queryMasses, referenceMasses,
                    queryDensities, referenceDensities,
                    f, f,
                    domain, adjacency, neighborhood, simulationState,       
                    nx, targetNumNeighbors, dim, device, nSamples, warpOnly
                )
                measurements.extend(interpolationResults)
            with record_function("Gradient Operations"):
                gradientResults = benchmarkOperation(
                    WarpOperation.Gradient, GradientScheme.Difference, LaplacianScheme.Naive,
                    queryPositions, referencePositions,
                    querySupports, referenceSupports,
                    queryMasses, referenceMasses,
                    queryDensities, referenceDensities,
                    f, f,
                    domain, adjacency, neighborhood, simulationState,
                    nx, targetNumNeighbors, dim, device, nSamples, warpOnly
                )
                measurements.extend(gradientResults)
            with record_function("Laplacian Operations"):
                LaplacianResults = benchmarkOperation(
                    WarpOperation.Laplacian, GradientScheme.Difference, LaplacianScheme.Brookshaw,
                    queryPositions, referencePositions,
                    querySupports, referenceSupports,
                    queryMasses, referenceMasses,
                    queryDensities, referenceDensities,
                    f, f,
                    domain, adjacency, neighborhood, simulationState,
                    nx, targetNumNeighbors, dim, device, nSamples, warpOnly
                )
                measurements.extend(LaplacianResults)

            f_vector = torch.randn((queryPositions.shape[0], queryPositions.shape[1]), device = queryPositions.device, dtype = queryPositions.dtype)
            with record_function("Divergence Operations"):
                divergenceResults = benchmarkOperation(
                    WarpOperation.Divergence, GradientScheme.Difference, LaplacianScheme.Brookshaw,
                    queryPositions, referencePositions,
                    querySupports, referenceSupports,
                    queryMasses, referenceMasses,
                    queryDensities, referenceDensities,
                    f_vector, f_vector,
                    domain, adjacency, neighborhood, simulationState,
                    nx, targetNumNeighbors, dim, device, nSamples, warpOnly
                )
                measurements.extend(divergenceResults)
            

            return measurements

In [11]:
nxs = [16, 32, 64, 128, 256, 512, 1024]

numParticles = np.logspace(10, 16, num=32, dtype=int, base = 2)
print(numParticles)
nxs = [int(np.sqrt(num)) for num in numParticles]

targetNumNeighbors = 50
dims = [2]
devices = ['cuda'] if torch.cuda.is_available() else ['cpu']
# devices = ['cpu']
all_measurements = []
nSamples = 16

warpOnly = False

_ = benchmark(nxs[0], targetNumNeighbors, dims[0], devices[0], nSamples , warpOnly)


[ 1024  1171  1339  1531  1751  2002  2290  2619  2995  3425  3916  4479
  5122  5857  6698  7660  8760 10018 11456 13101 14982 17133 19593 22406
 25623 29301 33508 38319 43821 50113 57308 65536]
Module sphWarpCore.radiusSearch.wp_compactHash e2c9126 load on device 'cuda:0' (block_dim=256) ...
            continue  # No particles in this cell

        dist_sq += dx * dx

                    count += 1

                    count += 1

            continue  # No particles in this cell

                    edge_index += 1

                    edge_index += 1

            hashValue += wp.uint32(cellIndex[d] * primes[d])

    Compile CUDA (arch=120, mode=release, block_dim=256) ...
Warning #20282-D: incompatible precompiled header (PCH) heap allocation address from PCH file "/home/lu26029/.cache/warp/1.12.0/wp_sphWarpCore.radiusSearch.wp_compactHash_e2c9126.pch".  This usually occurs if the PCH file was not created by the same dynamic instance of the NVRTC library, or if the NVRTC PCH heap 

Benchmarking [warp] with 1024 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/16 [00…

Benchmarking [diffSPH] with 1024 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/16 …

Benchmarking [warp] with 1024 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/16 [00…

[Kernel.add_overload] Creating new overload for computeSPHInterpolation_Kernel: a1v2f4_a1v2f4_a1f4_a1f4_a1f4_a1f4_a1f4_a1f4_a1f4_a1f4_a1f4_a1f4_a1b_u4_i4_a1i8_a1i4_a1i4_b_i4_a1i4_a1i4_b_a1f4_a1f4_b_a1f4_a1v2f4_a1f4
Module sphWarpCore.operations.wp_interpolate d6df463 load on device 'cuda:0' (block_dim=256) ...
        f_interpolated += fv * vj * w_ij

    Compile CUDA (arch=120, mode=release, block_dim=256) ...
Warning #20282-D: incompatible precompiled header (PCH) heap allocation address from PCH file "/home/lu26029/.cache/warp/1.12.0/wp___main___63bdc91.pch".  This usually occurs if the PCH file was not created by the same dynamic instance of the NVRTC library, or if the NVRTC PCH heap was freed or resized after the PCH file had been created (see documentation for details)

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

"wp_sphWarpCore.operations.wp_interpolate_d6df463.cu": using precompiled header file "/home/lu26029/.cache/warp/1.12.0/wp_sphWarpCore

...

[Kernel.add_overload] Creating new overload for computeSPHLaplacianTensor_Kernel: a1v2f4_a1v2f4_a1f4_a1f4_a1f4_a1f4_a1f4_a1f4_a1v1f4_a1v1f4_a1f4_a1f4_a1b_u4_i4_i4_i4_b_a1i8_a1i4_a1i4_b_i4_i4_i4_i4_i4_a1i4_a1i4_b_a1m22f4_b_a1f4_a1f4_b_a1f4_a1f4_b_a1f4_a1v2f4_a1v2f4_a1m22f4_a1v1f4
Module sphWarpCore.operations.wp_laplacian e6b690e load on device 'cuda:0' (block_dim=256) ...
            proj += q_ij[base + k] * n_ij[k]

        dot += x_ij[d] * fq_ij[d]

            laplacian_result += laplacian_contribution

    Compile CUDA (arch=120, mode=release, block_dim=256) ...
Warning #20282-D: incompatible precompiled header (PCH) heap allocation address from PCH file "/home/lu26029/.cache/warp/1.12.0/wp___main___63bdc91.pch".  This usually occurs if the PCH file was not created by the same dynamic instance of the NVRTC library, or if the NVRTC PCH heap was freed or resized after the PCH file had been created (see documentation for details)

Remark: The warnings can be suppressed with "-diag-sup

Benchmarking [diffSPH] with 1024 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/16 …

Benchmarking [warp] with 1024 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/16 [00…

[Kernel.add_overload] Creating new overload for computeSPHDivergenceTensor_Kernel: a1v2f4_a1v2f4_a1f4_a1f4_a1f4_a1f4_a1f4_a1f4_a1v2f4_a1v2f4_a1f4_a1f4_a1b_u4_i4_i4_b_a1i8_a1i4_a1i4_b_i4_i4_i4_b_i4_a1i4_a1i4_b_a1m22f4_b_a1f4_a1f4_b_a1f4_a1f4_b_a1f4_a1v2f4_a1v2f4_a1m22f4_a1v1f4
Module sphWarpCore.operations.wp_divergence 72ae8d6 load on device 'cuda:0' (block_dim=256) ...
                res[i] += fij[i + d * outputElements] * kernelGradient[d]

            grad_f_interpolated += divergenceProduct((fj + fi) * apparentVolume, kernelGradient, outputValue, flatOutputShape, dotMode)

    Compile CUDA (arch=120, mode=release, block_dim=256) ...
Warning #20282-D: incompatible precompiled header (PCH) heap allocation address from PCH file "/home/lu26029/.cache/warp/1.12.0/wp___main___63bdc91.pch".  This usually occurs if the PCH file was not created by the same dynamic instance of the NVRTC library, or if the NVRTC PCH heap was freed or resized after the PCH file had been created (see documenta

Benchmarking [diffSPH] with 1024 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/16 …

In [12]:

from torch.profiler import profile, record_function, ProfilerActivity

_ = benchmark(256, targetNumNeighbors, dims[0], devices[0], 8, False)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof:
    _ = benchmark(256, targetNumNeighbors, dims[0], devices[0], 8, False)


    
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))

prof.export_chrome_trace("profile.json")

Benchmarking [warp] with 65536 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/8 [00…

Benchmarking [diffSPH] with 65536 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/8 …

Benchmarking [warp] with 65536 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/8 [00…

Benchmarking [diffSPH] with 65536 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/8 …

Benchmarking [warp] with 65536 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/8 [00…

...

Benchmarking [warp] with 65536 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/8 [00…

Benchmarking [diffSPH] with 65536 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/8 …

Benchmarking [warp] with 65536 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/8 [00…

Benchmarking [diffSPH] with 65536 particles, target neighbors: 50, dim: 2, device: cuda:   0%|          | 0/8 …

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
Total benchmark for 65536 particles, target neighbor...         0.08%     565.856us        99.85%     708.089ms     708.089ms       0.000us         0.00%     442.554ms     442.554ms             1  
                                  cudaDeviceSynchronize        52.91%     375.225ms        52.91%     375.225ms       2.247ms       0.000us         0.00%       0.000us       0.000us           167  
         